# Qwen2.5-Math-1.5B LoRA Fine-tuning pentru BAC Matematica

**Setup Kaggle:**
- GPU T4 (16GB VRAM) sau P100
- Model: `Qwen/Qwen2.5-Math-1.5B`
- Metoda: LoRA (rank=16, alpha=32)
- Date: 433 train / 54 val / 55 test (542 exercitii, format ChatML)

**Inainte de a rula:**
1. Add Data → cauta `qwen25-math` si `bac-prep-data` → adauga ambele
2. Seteaza GPU: Settings → Accelerator → GPU T4 x2 (sau T4 x1)
3. Run All (nu trebuie sa editezi nimic!)

## 1. Instalare dependinte

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    accelerate>=0.30.0 bitsandbytes>=0.43.0 datasets>=2.19.0 \
    wandb scipy sentencepiece

## 2. Verificare GPU

In [ ]:
import torch
import os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU nu e disponibil! Seteaza Accelerator -> GPU in Kaggle Settings.")

## 3. Detectare automata path-uri dataset

Cauta automat fisierele in `/kaggle/input/` (nu trebuie sa editezi nimic).

In [ ]:
import json
import os
from pathlib import Path

def find_file(root, filename):
    """Cauta un fisier recursiv in /kaggle/input/"""
    for dirpath, dirnames, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    raise FileNotFoundError(f"Nu gasesc {filename} in {root}! Verifica ca ai adaugat dataset-ul.")

# Auto-detect paths
TRAIN_PATH = find_file("/kaggle/input", "train.jsonl")
VAL_PATH = find_file("/kaggle/input", "val.jsonl")
TEST_PATH = find_file("/kaggle/input", "test.jsonl")

print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

def load_jsonl(path):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"\nTrain: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")
print(f"Test:  {len(test_data)} samples")
print(f"\nSample:\n{train_data[0]['text'][:300]}...")

## 4. Incarcare model Qwen2.5-Math-1.5B + Quantizare 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Auto-detect model path (cauta config.json in /kaggle/input/)
def find_model_dir(root):
    """Gaseste directorul modelului cautand config.json"""
    for dirpath, dirnames, filenames in os.walk(root):
        if "config.json" in filenames and "tokenizer.json" in filenames:
            return dirpath
    raise FileNotFoundError("Nu gasesc modelul! Verifica ca ai adaugat dataset-ul qwen25-math.")

MODEL_NAME = find_model_dir("/kaggle/input")
print(f"Model gasit: {MODEL_NAME}")

# Quantizare 4-bit (NF4) - reduce VRAM de la ~6GB la ~1.5GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Incarc modelul...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
    local_files_only=True,
)

# Qwen foloseste <|endoftext|> ca pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model incarcat! Vocab size: {len(tokenizer)}")
print(f"Parametri: {model.num_parameters():,}")
print(f"VRAM folosit: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Configurare LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Pregatim modelul quantizat pentru antrenament
model = prepare_model_for_kbit_training(model)

# Configurare LoRA - aceleasi target_modules ca in lora_config.py
lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # Scaling factor (2x rank)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Ar trebui sa fie ~0.5-1% din total

## 6. Pregatire Dataset

In [ ]:
from datasets import Dataset

MAX_SEQ_LEN = 512

def tokenize_sample(sample):
    """Tokenizeaza un sample ChatML."""
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

# Convertim in HuggingFace Dataset
train_dataset = Dataset.from_list(train_data).map(tokenize_sample, remove_columns=["text"])
val_dataset = Dataset.from_list(val_data).map(tokenize_sample, remove_columns=["text"])

# Setam format pytorch
train_dataset.set_format("torch")
val_dataset.set_format("torch")

print(f"Train tokenizat: {len(train_dataset)} samples")
print(f"Val tokenizat:   {len(val_dataset)} samples")

# Verificam lungimea medie (inainte de padding)
avg_len = sum(1 for x in train_dataset if x["attention_mask"].sum() > 0)
print(f"Toate sample-urile au lungime {MAX_SEQ_LEN} (padded)")

## 7. Antrenament

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "/kaggle/working/qwen-bac-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epoci si batch
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,

    # Precision - no mixed precision (modelul e deja quantizat 4-bit)
    fp16=False,
    bf16=False,

    # Evaluare si salvare
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=10,
    logging_first_step=True,
    report_to="none",

    # Optimizare memorie
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,

    # Altele
    seed=42,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Incep antrenamentul...")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Total steps: ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * int(training_args.num_train_epochs)}")

In [ ]:
# Lanseaza antrenamentul
train_result = trainer.train()

# Afiseaza metrici
print(f"\n{'='*60}")
print(f"Training complet!")
print(f"  Train loss final:  {train_result.metrics['train_loss']:.4f}")
print(f"  Training runtime:  {train_result.metrics['train_runtime']:.0f}s")
print(f"{'='*60}")

## 8. Evaluare pe Test Set

In [ ]:
# Evaluare pe test set
test_dataset = Dataset.from_list(test_data).map(tokenize_sample, remove_columns=["text"])

test_results = trainer.evaluate(eval_dataset=test_dataset)
print(f"\nTest loss: {test_results['eval_loss']:.4f}")
print(f"Test perplexity: {torch.exp(torch.tensor(test_results['eval_loss'])):.2f}")

## 9. Salvare adapter LoRA

In [ ]:
# Salvam DOAR adapter-ul LoRA (cateva MB, nu modelul intreg)
ADAPTER_PATH = "/kaggle/working/qwen-bac-lora/best_adapter"
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"Adapter salvat in {ADAPTER_PATH}")

# Verificam dimensiunea
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
)
print(f"Dimensiune adapter: {total_size / 1e6:.1f} MB")

## 10. Test inferenta

In [ ]:
# Test cu cateva exercitii
test_questions = [
    "Rezolva ecuatia: 2x + 3 = 7",
    "Calculati derivata functiei f(x) = x^3 - 6x^2 + 9x + 1",
    "Calculati limita: lim(x->inf) (x^2 + 3x) / (2x^2 - 1)",
    "Calculati det[[3,1],[2,4]]",
]

SYSTEM_PROMPT = (
    "Ești un asistent de matematică specializat pe exerciții BAC. "
    "Rezolvă exercițiul pas cu pas, explicând fiecare etapă."
)

model.eval()
for q in test_questions:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{q}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_k=30,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>")[0],
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    # Taiem la <|im_end|>
    if "<|im_end|>" in response:
        response = response[:response.index("<|im_end|>")]
    
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response.strip()}")

## 11. Download adapter

Descarca adapter-ul si pune-l in `ai/finetune/adapters/best_adapter/` local.

In [ ]:
# Comprima adapter-ul pentru download
!cd /kaggle/working && tar -czf qwen-bac-lora-adapter.tar.gz qwen-bac-lora/best_adapter/
print("\nDescarca: qwen-bac-lora-adapter.tar.gz")
print("Pune continutul in: ai/finetune/adapters/best_adapter/")

## 12. (Optional) Push adapter pe HuggingFace Hub

In [ ]:
# Decomment daca vrei sa uploadezi pe HuggingFace
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
#
# model.push_to_hub("YOUR_USERNAME/qwen-bac-math-lora")
# tokenizer.push_to_hub("YOUR_USERNAME/qwen-bac-math-lora")